What Stage 5 does and why it's structured this wayStage 5 converts the continuous density field from SIMP into a manufacturable STL file. This is harder than it sounds — the density field lives on an unstructured tetrahedral mesh, but marching cubes (the standard isosurface extraction algorithm) expects a structured voxel grid. The bridge between these two representations is the core problem this stage solves.The data flow for this stage is:outputs/meshes/<name>_stage04.json      ← handoff from Stage 4
outputs/meshes/<name>_density.xdmf     ← density field on tet mesh
        ↓
voxelization: tet mesh → regular grid   ← scattered interpolation
marching cubes (scikit-image)           ← isosurface at rho=0.5
mesh repair (trimesh)                   ← fill holes, fix normals, remove islands
        ↓
outputs/stl/<name>_optimized.stl        ← final export
outputs/reports/<name>_stl_*.png        ← before/after renders
outputs/meshes/<name>_stage05.json      ← pipeline completion recordThere is no src/ module for this stage — the logic is self-contained enough that splitting it would add indirection without benefit. Everything lives in the notebook cells, backed by scikit-image and trimesh which are already in the container.

Cell 0 — Parameters (tag: parameters)

In [1]:
# Cell 0 — tagged: parameters
import os
os.chdir("/workspace")

import sys
sys.path.insert(0, "/workspace")

STAGE04_HANDOFF  = None    # auto-detect if None
DENSITY_THRESHOLD = 0.5    # isosurface level — rho > this is "solid"
                            # raise to 0.6 for heavier/safer result
                            # lower to 0.4 for lighter/more aggressive
VOXEL_RESOLUTION  = 100    # grid divisions along longest axis
                            # 100 = fast preview, 200 = production quality
                            # memory scales as resolution^3 — don't exceed 300
SMOOTH_ITERATIONS = 5       # Laplacian smoothing passes on final mesh
                            # 0 = raw marching cubes, 10+ = very rounded
REMOVE_ISLANDS    = True    # drop disconnected components smaller than 1%
                            # of total mesh volume
MIN_ISLAND_FRAC   = 0.01   # island removal threshold (fraction of total volume)

Cell 1 — Load Stage 4 handoff

In [2]:
# Cell 1 — Read handoff from 04_simp_optimization.ipynb
import json
import numpy as np
from pathlib import Path

if STAGE04_HANDOFF is None:
    candidates = sorted(Path("outputs/meshes").glob("*_stage04.json"))
    if not candidates:
        raise FileNotFoundError(
            "No stage04 handoff in outputs/meshes/. "
            "Run 04_simp_optimization.ipynb first."
        )
    handoff_path = candidates[-1]
else:
    handoff_path = Path(STAGE04_HANDOFF)

handoff = json.loads(handoff_path.read_text())
print(f"Loaded handoff: {handoff_path}")
print(json.dumps(handoff, indent=2))

part_name      = handoff["part_name"]
density_path   = Path(handoff["density_path"])
xdmf_path      = Path(handoff["xdmf_path"])
config         = handoff["config"]
final_vol_frac = handoff["final_volume_frac"]

assert density_path.exists(), f"Density XDMF not found: {density_path}"
assert xdmf_path.exists(),    f"Mesh XDMF not found: {xdmf_path}"

print(f"\nDensity field: {density_path}")
print(f"Final volume fraction: {final_vol_frac:.3f}")
print(f"Filter radius used:    {config['filter_radius']} mm")
print(f"Threshold to apply:    {DENSITY_THRESHOLD}")

Loaded handoff: outputs/meshes/base_part_stage04.json
{
  "stage": "04_simp",
  "part_name": "base_part",
  "density_path": "outputs/meshes/base_part_density.xdmf",
  "xdmf_path": "outputs/meshes/base_part.xdmf",
  "n_iterations": 34,
  "converged": true,
  "final_compliance": 407372.50067206717,
  "final_volume_frac": 0.7623995584744789,
  "duration_s": 28.154,
  "config": {
    "volume_fraction": 0.4,
    "penal": 3.0,
    "filter_radius": 0.3,
    "convergence_tol": 0.001
  },
  "material": {
    "youngs_modulus_pa": 210000000000.0,
    "poissons_ratio": 0.3,
    "name": "custom_210.0GPa"
  },
  "load_hints": {
    "primary_face": "top",
    "load_magnitude_n": 1000.0
  }
}

Density field: outputs/meshes/base_part_density.xdmf
Final volume fraction: 0.762
Filter radius used:    0.3 mm
Threshold to apply:    0.5


Cell 2 — Load density field and mesh geometry

In [3]:
# Cell 2 — Extract density values and node coordinates from XDMF
#
# dolfinx stores DG0 functions indexed by element, not by node.
# We need: element centroids (where density is defined) + density values.
# These feed into the scattered interpolation step in Cell 3.

from mpi4py import MPI
import dolfinx
import dolfinx.fem as fem
import ufl
from dolfinx.io import XDMFFile

comm = MPI.COMM_WORLD

# Load the mesh
with XDMFFile(comm, str(xdmf_path), "r") as xdmf:
    domain = xdmf.read_mesh(name="Grid")

# Load density function onto a DG0 space
W = fem.functionspace(domain, ("DG", 0))

# read_function removed in dolfinx v0.9.0 — read directly from HDF5
import h5py
h5_path = str(density_path).replace(".xdmf", ".h5")
with h5py.File(h5_path, "r") as f:
    density_data = f["Function/density/0"][:]

# Flatten from (n_elements, 1) to (n_elements,)
rho_values = density_data.ravel()

points = domain.geometry.x

# Compute element centroids
tdim = domain.topology.dim
domain.topology.create_connectivity(tdim, 0)
conn = domain.topology.connectivity(tdim, 0)
n_elem = W.dofmap.index_map.size_local
tets = np.array([conn.links(i) for i in range(n_elem)])
centroids = points[tets].mean(axis=1)   # shape: (n_elements, 3)

print(f"Mesh nodes:         {len(points):,}")
print(f"Elements:           {n_elem:,}")
print(f"Density range:      [{rho_values.min():.4f}, {rho_values.max():.4f}]")
print(f"Elements > thresh:  "
      f"{(rho_values > DENSITY_THRESHOLD).sum():,} "
      f"({100*(rho_values > DENSITY_THRESHOLD).mean():.1f}%)")

bb_min = points.min(axis=0)
bb_max = points.max(axis=0)
print(f"Bounding box:       "
      f"X=[{bb_min[0]:.1f}, {bb_max[0]:.1f}] "
      f"Y=[{bb_min[1]:.1f}, {bb_max[1]:.1f}] "
      f"Z=[{bb_min[2]:.1f}, {bb_max[2]:.1f}] mm")

Mesh nodes:         16,052
Elements:           66,912
Density range:      [0.0010, 1.0000]
Elements > thresh:  66,148 (98.9%)
Bounding box:       X=[-2.0, 98.0] Y=[-2.0, 58.0] Z=[0.0, 20.0] mm


Cell 3 — Voxelize density field

In [4]:
# Cell 3 — Interpolate scattered density onto a regular voxel grid
#
# Marching cubes needs a 3D array — a regular grid of density values.
# We get there via scipy's NearestNDInterpolator:
#   - Fast: O(n log n) KD-tree build, O(m) query for m voxels
#   - Nearest-neighbor is appropriate here because DG0 density is
#     piecewise constant — linear interpolation would artificially
#     smooth the threshold boundary before marching cubes does it.
#
# Memory estimate: resolution^3 * 8 bytes
#   100^3 =  8 MB  — fine
#   200^3 = 64 MB  — fine
#   300^3 =216 MB  — watch WSL2 memory
#   400^3 =512 MB  — risky, reduce if OOM

from scipy.interpolate import NearestNDInterpolator
import numpy as np

# Build regular grid
extent = bb_max - bb_min
longest_axis = extent.max()
voxel_size = longest_axis / VOXEL_RESOLUTION

nx = max(2, int(np.ceil(extent[0] / voxel_size)))
ny = max(2, int(np.ceil(extent[1] / voxel_size)))
nz = max(2, int(np.ceil(extent[2] / voxel_size)))

print(f"Voxel size:   {voxel_size:.3f} mm")
print(f"Grid shape:   {nx} x {ny} x {nz} "
      f"({nx*ny*nz:,} voxels, "
      f"{nx*ny*nz*8/1e6:.1f} MB)")

xs = np.linspace(bb_min[0], bb_max[0], nx)
ys = np.linspace(bb_min[1], bb_max[1], ny)
zs = np.linspace(bb_min[2], bb_max[2], nz)
gx, gy, gz = np.meshgrid(xs, ys, zs, indexing='ij')
grid_points = np.column_stack([gx.ravel(), gy.ravel(), gz.ravel()])

# Interpolate — NearestND is fast and appropriate for DG0 fields
print("Interpolating density field onto voxel grid...")
interp = NearestNDInterpolator(centroids, rho_values)
density_grid = interp(grid_points).reshape(nx, ny, nz)

print(f"Grid density range: [{density_grid.min():.4f}, {density_grid.max():.4f}]")
print(f"Voxels above threshold: "
      f"{(density_grid > DENSITY_THRESHOLD).sum():,} "
      f"({100*(density_grid > DENSITY_THRESHOLD).mean():.1f}%)")

Voxel size:   1.000 mm
Grid shape:   100 x 60 x 20 (120,000 voxels, 1.0 MB)
Interpolating density field onto voxel grid...
Grid density range: [0.0010, 1.0000]
Voxels above threshold: 103,881 (86.6%)


Cell 4 — Marching cubes

In [5]:
# Cell 4 — Extract isosurface via marching cubes
from skimage.measure import marching_cubes
import numpy as np

print(f"Running marching cubes at threshold {DENSITY_THRESHOLD}...")

# Invert density field so marching cubes extracts SOLID regions (rho > threshold)
# rather than void regions. Without inversion the surface of void pockets
# is extracted instead of the solid material boundary.
verts_vox, faces, normals, values = marching_cubes(
    1.0 - density_grid,
    level=1.0 - DENSITY_THRESHOLD,
    spacing=(1.0, 1.0, 1.0),
    gradient_direction='descent',
    allow_degenerate=False,
)

# Rescale from voxel index space to mm
verts_mm = verts_vox * voxel_size + bb_min

print(f"Vertices:  {len(verts_mm):,}")
print(f"Triangles: {len(faces):,}")
print(f"Bounds:    "
      f"X=[{verts_mm[:,0].min():.1f}, {verts_mm[:,0].max():.1f}] "
      f"Y=[{verts_mm[:,1].min():.1f}, {verts_mm[:,1].max():.1f}] "
      f"Z=[{verts_mm[:,2].min():.1f}, {verts_mm[:,2].max():.1f}] mm")

if len(verts_mm) == 0:
    raise ValueError(
        f"Marching cubes produced no geometry at threshold {DENSITY_THRESHOLD}. "
        f"Try lowering DENSITY_THRESHOLD."
    )

Running marching cubes at threshold 0.5...
Vertices:  18,627
Triangles: 35,222
Bounds:    X=[-2.0, 97.0] Y=[-2.0, 57.0] Z=[0.0, 19.0] mm


Cell 5 — Mesh repair with trimesh

In [6]:
# Cell 5 — Repair and clean the raw marching cubes output
#
# Raw marching cubes output typically has:
#   - Duplicate vertices (MC generates one per triangle, not shared)
#   - Non-manifold edges at voxel boundaries
#   - Small disconnected islands (numerical noise from filter edges)
#   - Inconsistent face winding (normals pointing inward on some faces)
#
# trimesh handles all of these. The order of operations matters:
#   1. Merge vertices first (required for watertight check)
#   2. Remove islands before smoothing (don't waste iterations on trash)
#   3. Fix normals after island removal (island removal can flip winding)
#   4. Smooth last (operates on the clean mesh)

import trimesh
import numpy as np

print("Building trimesh...")
mesh = trimesh.Trimesh(
    vertices=verts_mm,
    faces=faces,
    process=False,    # disable auto-processing — we do it manually below
)

print(f"Raw:  {len(mesh.vertices):,} verts, {len(mesh.faces):,} faces")
print(f"Watertight (raw): {mesh.is_watertight}")

# Step 1: merge duplicate vertices
mesh.merge_vertices()
print(f"After merge:  {len(mesh.vertices):,} verts, {len(mesh.faces):,} faces")

# Step 2: remove disconnected islands
if REMOVE_ISLANDS:
    components = mesh.split(only_watertight=False)
    if len(components) > 1:
        volumes = [abs(c.volume) for c in components]
        total_vol = sum(volumes)
        kept = [c for c, v in zip(components, volumes)
                if v / total_vol >= MIN_ISLAND_FRAC]
        dropped = len(components) - len(kept)
        if kept:
            mesh = trimesh.util.concatenate(kept)
            print(f"Removed {dropped} island(s) below "
                  f"{MIN_ISLAND_FRAC*100:.1f}% volume threshold")
        else:
            print("⚠ All components below island threshold — keeping largest")
            mesh = components[np.argmax(volumes)]
    else:
        print("No disconnected islands found")

# Step 3: fix face winding and normals
trimesh.repair.fix_normals(mesh)
trimesh.repair.fix_winding(mesh)

# Step 4: fill small holes (marching cubes boundary artifacts)
trimesh.repair.fill_holes(mesh)

# Step 5: Laplacian smoothing
if SMOOTH_ITERATIONS > 0:
    trimesh.smoothing.filter_laplacian(
        mesh,
        lamb=0.5,
        iterations=SMOOTH_ITERATIONS,
        implicit_time_integration=False,
        volume_constraint=True,   # preserve volume during smoothing
    )
    print(f"Applied {SMOOTH_ITERATIONS} Laplacian smoothing iterations")

print(f"\nFinal: {len(mesh.vertices):,} verts, {len(mesh.faces):,} faces")
print(f"Watertight: {mesh.is_watertight}")
print(f"Volume:     {abs(mesh.volume):.2f} mm³")
print(f"Area:       {mesh.area:.2f} mm²")

if not mesh.is_watertight:
    print("⚠ Mesh is not watertight — may cause issues in slicers.")
    print("  Options:")
    print("  1. Increase VOXEL_RESOLUTION (more geometry detail)")
    print("  2. Increase SMOOTH_ITERATIONS (closes small gaps)")
    print("  3. Increase FILTER_RADIUS in Stage 4 (removes thin features)")

Building trimesh...
Raw:  18,627 verts, 35,222 faces
Watertight (raw): False
After merge:  18,627 verts, 35,222 faces
Removed 57 island(s) below 1.0% volume threshold
Applied 5 Laplacian smoothing iterations

Final: 16,631 verts, 31,726 faces
Watertight: False
Volume:     8247.77 mm³
Area:       8377.88 mm²
⚠ Mesh is not watertight — may cause issues in slicers.
  Options:
  1. Increase VOXEL_RESOLUTION (more geometry detail)
  2. Increase SMOOTH_ITERATIONS (closes small gaps)
  3. Increase FILTER_RADIUS in Stage 4 (removes thin features)


Cell 6 — Export STL

In [7]:
# Cell 6 — Export final STL
from pathlib import Path
import trimesh

stl_dir  = Path("outputs/stl")
stl_dir.mkdir(parents=True, exist_ok=True)
stl_path = stl_dir / f"{part_name}_optimized.stl"

# Binary STL — smaller file, faster to write and read in slicers
mesh.export(str(stl_path), file_type="stl")

stl_size_kb = stl_path.stat().st_size / 1024
print(f"STL exported:  {stl_path}")
print(f"File size:     {stl_size_kb:.1f} KB")
print(f"Triangles:     {len(mesh.faces):,}")
print(f"Watertight:    {mesh.is_watertight}")
print(f"Volume:        {abs(mesh.volume):.2f} mm³")

# Compare volume to original bounding box — gives a feel for material removed
bb_volume = float(np.prod(bb_max - bb_min))
print(f"BB volume:     {bb_volume:.2f} mm³")
print(f"Fill ratio:    {abs(mesh.volume)/bb_volume*100:.1f}% "
      f"(target was {handoff['config']['volume_fraction']*100:.0f}%)")

STL exported:  outputs/stl/base_part_optimized.stl
File size:     1549.2 KB
Triangles:     31,726
Watertight:    False
Volume:        8247.77 mm³
BB volume:     120000.00 mm³
Fill ratio:    6.9% (target was 40%)


Cell 7 — Before/after renders

In [8]:
# Cell 7 — Side-by-side render: original STL vs optimized STL
import pyvista as pv
from pathlib import Path

pv.OFF_SCREEN = True
report_dir = Path("outputs/reports")

# Original geometry from Stage 1
original_stl = Path(handoff.get(
    "stl_path",
    f"outputs/meshes/{part_name}.stl"
))

pl = pv.Plotter(shape=(1, 2), window_size=(1600, 600))

if original_stl.exists():
    orig_mesh = pv.read(str(original_stl))
    pl.subplot(0, 0)
    pl.add_mesh(orig_mesh, color="lightsteelblue", show_edges=False, opacity=0.9)
    pl.add_text(
        f"Original\n{orig_mesh.n_cells:,} faces",
        font_size=10,
    )
    pl.view_isometric()
else:
    pl.subplot(0, 0)
    pl.add_text("Original STL not found", font_size=10)

# Optimized
opt_mesh = pv.read(str(stl_path))
pl.subplot(0, 1)
pl.add_mesh(opt_mesh, color="coral", show_edges=False, opacity=0.9)
pl.add_text(
    f"Optimized (vf={final_vol_frac:.2f})\n{opt_mesh.n_cells:,} faces",
    font_size=10,
)
pl.view_isometric()

compare_png = report_dir / f"{part_name}_before_after.png"
pl.screenshot(str(compare_png))
pl.close()
print(f"Before/after render: {compare_png}")

# Wireframe of optimized — useful for checking surface quality
pl2 = pv.Plotter()
pl2.add_mesh(opt_mesh, style="wireframe", color="steelblue", line_width=0.5)
pl2.view_isometric()
wire_png = report_dir / f"{part_name}_stl_wireframe.png"
pl2.screenshot(str(wire_png))
pl2.close()
print(f"Wireframe render:    {wire_png}")

Before/after render: outputs/reports/base_part_before_after.png
Wireframe render:    outputs/reports/base_part_stl_wireframe.png


Cell 8 — Pipeline completion record

In [9]:
# Cell 8 — Write final stage05 record and pipeline summary
#
# This is the terminal handoff — no further stages consume it.
# It's a complete audit record of the full pipeline run:
# parameters, timings, and output paths for every stage.

import json
from pathlib import Path
from datetime import datetime

# Gather all stage handoffs for the summary
stage_handoffs = {}
for stage_json in sorted(Path("outputs/meshes").glob(f"{part_name}_stage0*.json")):
    stage_data = json.loads(stage_json.read_text())
    stage_handoffs[stage_data["stage"]] = stage_data

record = {
    "stage":          "05_stl_export",
    "part_name":      part_name,
    "completed_at":   datetime.utcnow().isoformat() + "Z",
    "stl_path":       str(stl_path),
    "stl_size_kb":    round(stl_size_kb, 2),
    "mesh": {
        "vertices":   len(mesh.vertices),
        "triangles":  len(mesh.faces),
        "watertight": mesh.is_watertight,
        "volume_mm3": round(abs(mesh.volume), 3),
        "area_mm2":   round(mesh.area, 3),
    },
    "export_config": {
        "density_threshold": DENSITY_THRESHOLD,
        "voxel_resolution":  VOXEL_RESOLUTION,
        "smooth_iterations": SMOOTH_ITERATIONS,
        "remove_islands":    REMOVE_ISLANDS,
    },
    "pipeline_summary": {
        s: {
            "duration_s": v.get("duration_s"),
        }
        for s, v in stage_handoffs.items()
    },
    "optimization": {
        "volume_fraction":   final_vol_frac,
        "n_iterations":      handoff["n_iterations"],
        "converged":         handoff["converged"],
        "final_compliance":  handoff["final_compliance"],
    },
}

record_path = Path("outputs/meshes") / f"{part_name}_stage05.json"
record_path.write_text(json.dumps(record, indent=2))

print("═" * 60)
print("  PIPELINE COMPLETE")
print("═" * 60)
print(f"  Part:          {part_name}")
print(f"  STL:           {stl_path}")
print(f"  Watertight:    {mesh.is_watertight}")
print(f"  Volume:        {abs(mesh.volume):.2f} mm³")
print(f"  Triangles:     {len(mesh.faces):,}")
print(f"  Material saved:{(1 - final_vol_frac)*100:.1f}%")
print("─" * 60)
print("  Stage durations:")
for stage, data in stage_handoffs.items():
    dur = data.get("duration_s", "—")
    print(f"    {stage:<20} {dur}s")
print("═" * 60)
print(f"\n  Full record: {record_path}")

════════════════════════════════════════════════════════════
  PIPELINE COMPLETE
════════════════════════════════════════════════════════════
  Part:          base_part
  STL:           outputs/stl/base_part_optimized.stl
  Watertight:    False
  Volume:        8247.77 mm³
  Triangles:     31,726
  Material saved:23.8%
────────────────────────────────────────────────────────────
  Stage durations:
    01_geometry          0.251s
    02_meshing           —s
    03_fea               —s
    04_simp              28.154s
════════════════════════════════════════════════════════════

  Full record: outputs/meshes/base_part_stage05.json


Common failure modes in Stage 5 and how to fix them
Zero geometry from marching cubes — DENSITY_THRESHOLD is too high for the actual density range. Check density_grid.max() in Cell 3 output — if it's below your threshold the level set doesn't exist. Lower DENSITY_THRESHOLD or re-run Stage 4 with a lower volume_fraction.
Spiky/disconnected STL — filter_radius in Stage 4 was too small relative to target_element_size. The fix is in scad/params.json (increase target_element_size) or in Stage 4 Cell 0 (increase FILTER_RADIUS). Re-run from Stage 4.
Not watertight after repair — usually thin features at voxel resolution. Either increase VOXEL_RESOLUTION (200+) or increase SMOOTH_ITERATIONS (10+). If it still won't close, the optimized geometry has genuine holes — increase filter_radius in Stage 4 to enforce a larger minimum feature size.
Volume ratio far from target — if fill_ratio in Cell 6 is significantly different from volume_fraction from Stage 4, the threshold is cutting differently than expected. This is normal for thresholds other than 0.5 — the SIMP density field is not uniformly distributed. Adjust DENSITY_THRESHOLD rather than re-running the optimizer.